In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_reindex_small.h5ad')
adata.obsm = tmp_adata.obsm.copy()

In [ ]:
clusters_path = f'{projdir}/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['clusters_v7']
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata, color=['coarse', 'time_num'])

In [ ]:
tmp_groups = [f'Fail2_{x}' for x in range(1,6)]
sc.pl.umap(adata, color=['cluster'], groups=tmp_groups)

### fail2 genes

In [ ]:
degs_path = f'{projdir}/csv/degs/justin_degs_v7.csv'
degs = pd.read_csv(degs_path, sep='\t', index_col=0)
degs = degs[degs['p_val_adj'] < 0.01]
degs.shape

In [ ]:
degs.head()

In [ ]:
s = 'Fail2_1'
tmp_degs = degs[degs['cluster_v7'] == s].sort_values(by=['avg_log2FC'], ascending=False)
tmp_degs.shape

In [ ]:
tmp_degs.head()

In [ ]:
tmp_degs.index[:50]

In [ ]:
tmp_groups = [f'Fail2_{x}' for x in range(1,6)]

In [ ]:
sc.pl.umap(adata, color=['cluster'], groups=tmp_groups)

In [ ]:
sc.pl.umap(adata, color=['cluster'], groups=['Fail2_3'])

In [ ]:
s = 'Fail2_1'
adata.obs[adata.obs['cluster'] == s].shape

In [ ]:
genes = ['BST2', 'IFI27']
tmp_adata = adata.copy()
sc.pl.umap(tmp_adata, color=genes, ncols=2)

### Fail2_2

In [ ]:
s = 'Fail2_2'
tmp_degs = degs[degs['cluster_v7'] == s].sort_values(by=['avg_log2FC'], ascending=False)
tmp_genes = tmp_degs.index[:50]
tmp_genes

In [ ]:
genes = ['GRM1', 'GABRA2', 'PEX5L', 'SNTG1']
sc.pl.umap(adata, color=genes, ncols=2, vmax=2)

### fail2_3

In [ ]:
s = 'Fail2_3'
tmp_degs = degs[degs['cluster_v7'] == s].sort_values(by=['avg_log2FC'], ascending=False)
tmp_genes = tmp_degs.index[:50]
tmp_genes

In [ ]:
genes = ['NTRK2', 'SYBU', 'SNCAIP', 'GALNT13']
sc.pl.umap(adata, color=genes, ncols=2, vmax=2)

### fail2_5

In [ ]:
### not in the gene list so let's just do Fail2_5 vs Inter1 for now
s = 'Fail2_5'
tmp_degs = degs[degs['cluster_v7'] == s].sort_values(by=['avg_log2FC'], ascending=False)
tmp_genes = tmp_degs.index[:50]
tmp_genes

In [ ]:
clusters[clusters['clusters_v7'] == 'Fail2_5'].shape

In [ ]:
tmp_degs = pd.read_csv(f'{projdir}/csv/degs/fail2/fail2.csv', sep='\t', index_col=0)
tmp_degs = tmp_degs[tmp_degs['cluster'] == 'Fail2_5']
genes = tmp_degs.sort_values(by='logfoldchanges', ascending=False)[:50]['names']
genes.shape

In [ ]:
genes.values

In [ ]:
sc.pl.umap(adata, color=genes, ncols=5)

In [ ]:
genes = ['CCNA1', 'MBD3L2', 'TRIM48', 'KDM4E']
sc.pl.umap(adata, color=genes, ncols=2)

### get some IPS degs

In [ ]:
degs_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/degs/justin_degs_v7.csv'
degs = pd.read_csv(degs_path, sep='\t')
degs.shape

In [ ]:
tmp_degs = degs[degs['cluster_v7'] == 'IPS_3']
tmp_degs.shape

In [ ]:
tmp_degs.sort_values(by=['avg_log2FC'], ascending=False)['gene'][:50].values

In [ ]:
whitelist = ['NANOG', 'EPCAM', 'DNMT3B']
mask = [x in whitelist for x in tmp_degs['gene']]
tmp_degs[mask]

### final umap df

In [ ]:
genes1 = ['BST2', 'IFI27'] # Fail2_1
genes2 = ['GRM1', 'GABRA2', 'PEX5L', 'SNTG1'] # Fail2_2
genes3 = ['NTRK2', 'SYBU', 'SNCAIP', 'GALNT13'] # Fail2_3
genes4 = ['CCNA1', 'MBD3L2', 'TRIM48', 'KDM4E'] # Fail2_5
genes5 = ['NANOG', 'EPCAM', 'DNMT3B', 'TDGF1'] # IPS
genes = genes1 + genes2 + genes3 + genes4 + genes5

In [ ]:
tmp_adata = adata.copy()
indices = tmp_adata.var.index.get_indexer(genes)
for g,idx in zip(genes, indices):
    tmp_adata.obs[g] = np.ravel(tmp_adata[:,idx].X.toarray())

In [ ]:
umap_df = pd.DataFrame(index=tmp_adata.obs.index)
umap_df['umap0'] = tmp_adata.obsm['X_umap'][:,0]
umap_df['umap1'] = tmp_adata.obsm['X_umap'][:,1]

select = ['time_num', 'cluster'] + genes
for s in select:
    umap_df[s] = tmp_adata.obs[s]

In [ ]:
umap_df.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
umap_df.to_csv(f'{outdir}/multiome_rna_fail2_umap.csv', sep='\t')

### long-form for dot plot

In [ ]:
genes1 = ['BST2', 'IFI27'] # Fail2_1
genes2 = ['GRM1', 'GABRA2', 'PEX5L', 'SNTG1'] # Fail2_2
genes3 = ['NTRK2', 'SYBU', 'SNCAIP', 'GALNT13'] # Fail2_3
genes4 = ['CCNA1', 'MBD3L2', 'TRIM48', 'KDM4E'] # Fail2_5
genes5 = ['NANOG', 'EPCAM', 'DNMT3B', 'TDGF1'] # IPS
genes = genes1 + genes2 + genes3 + genes4 + genes5
to_plot = umap_df.melt(id_vars=['cluster'], value_vars=genes)
to_plot.head()

In [ ]:
print(genes)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
to_plot.to_csv(f'{outdir}/multiome_rna_fail2_dot_plot.csv', sep='\t')

In [ ]:
!date